## Backend de RAG

### Importar librerias y utilidades

In [1]:
import io
import os
from dotenv import load_dotenv
load_dotenv()
from azure.core.credentials import AzureKeyCredential
from azure.storage.blob import BlobServiceClient
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeResult
from openai import AzureOpenAI
import requests
import cupy as cp
import numpy as np
import ivf_search

ModuleNotFoundError: No module named 'azure'

### Importar variables de entorno y variables utilies para el procesamiento

In [13]:
AZURE_STORAGE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
DEPLOYMENT = "phi-4-mini-instruct"
DOCUMENT_INTELLIGENCE_ENDPOINT = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")
DOCUMENT_INTELLIGENCE_KEY = os.getenv("DOCUMENT_INTELLIGENCE_KEY")
AZURE_OPEN_API_KEY_EMMB = os.getenv('AZURE_OPEN_API_KEY_EMMB')
AZURE_ENDPOINT_EMMB = os.getenv('AZURE_ENDPOINT_EMMB')

client = AzureOpenAI(
    api_key= os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint= os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version= "2024-08-01-preview"  
)

client_emb = AzureOpenAI(
    api_key= os.environ["AZURE_OPEN_API_KEY_EMMB"],                      
    azure_endpoint= os.environ["AZURE_ENDPOINT_EMMB"],
    api_version= "2024-02-01"                                
)

#### Funcion para obtener texto de pdf 

In [14]:
def obtener_texto_de_blob_pdf(BLOB_NAME, CONTAINER_NAME):
    try:

        if not all([AZURE_STORAGE_CONNECTION_STRING, DOCUMENT_INTELLIGENCE_ENDPOINT, DOCUMENT_INTELLIGENCE_KEY]):
            raise ValueError("Faltan configurar variables en tu archivo .env. Por favor verifícalo.")

        print(f"Conectando a Blob Storage para descargar: {BLOB_NAME}...")
        blob_service_client = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
        blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=BLOB_NAME)
        

        pdf_bytes = blob_client.download_blob().readall()
        
    
        pdf_stream = io.BytesIO(pdf_bytes)
        
        print("Enviando el archivo a Azure AI Document Intelligence...")
        client = DocumentIntelligenceClient(
            endpoint=DOCUMENT_INTELLIGENCE_ENDPOINT, 
            credential=AzureKeyCredential(DOCUMENT_INTELLIGENCE_KEY)
        )
        
        poller = client.begin_analyze_document(
            model_id="prebuilt-layout",
            body=pdf_stream,
            content_type="application/pdf",
            output_content_format="markdown"
        )
        
        resultado: AnalyzeResult = poller.result()
        texto_final_string = resultado.content
        return texto_final_string

    except Exception as e:
        print(f"Ocurrió un error en el proceso: {e}")
        return None

In [15]:
text_pdf = obtener_texto_de_blob_pdf('7245fcea1c7bf0a995952dabe3da456d.pdf','pdf')
print(text_pdf)

Conectando a Blob Storage para descargar: 7245fcea1c7bf0a995952dabe3da456d.pdf...
Enviando el archivo a Azure AI Document Intelligence...
# A Tutorial on Principal Component Analysis

Jonathon Shlens*
Google Research
Mountain View, CA 94043

(Dated: April 7, 2014; Version 3.02)

Principal component analysis (PCA) is a mainstay of modern data analysis - a black box that is widely used
but (sometimes) poorly understood. The goal of this paper is to dispel the magic behind this black box. This
manuscript focuses on building a solid intuition for how and why principal component analysis works. This
manuscript crystallizes this knowledge by deriving from simple intuitions, the mathematics behind PCA. This
tutorial does not shy away from explaining the ideas informally, nor does it shy away from the mathematics. The
hope is that by addressing both aspects, readers of all levels will be able to gain a better understanding of PCA as
well as the when, the how and the why of applying this techni

### Funcion para dividir el texto en fragmentos

In [16]:
def recursive_text_splitter(text: str, max_chunk_size: int = 500, overlap_size: int = 50) -> list[str]:
    """
    Divide un texto en segmentos (chunks) para RAG sin usar librerías externas.
    Respeta la estructura semántica (párrafos, oraciones, palabras) y añade solapamiento.
    
    :param text: El string de texto completo a dividir.
    :param max_chunk_size: Límite máximo de caracteres por segmento.
    :param overlap_size: Cantidad de caracteres a duplicar entre segmentos adyacentes.
    :return: Lista de strings (chunks).
    """
    # 1. Definimos la jerarquía de separadores (de mayor a menor peso semántico)
    separators = ["\n\n", "\n", ". ", " ", ""]
    
    def _split_recursive(chunk_text: str, current_seps: list[str]) -> list[str]:
        # Caso base 1: Si el fragmento ya entra en el tamaño máximo, se devuelve tal cual
        if len(chunk_text) <= max_chunk_size:
            return [chunk_text]
            
        # Caso base 2: Si nos quedamos sin separadores, cortamos estrictamente por caracteres
        if not current_seps:
            return [chunk_text[i:i + max_chunk_size] for i in range(0, len(chunk_text), max_chunk_size)]
            
        sep = current_seps[0]
        splits = chunk_text.split(sep)
        
        chunks = []
        current_chunk = ""
        
        for part in splits:
            # Reconstruimos el texto añadiendo el separador que eliminó el .split()
            potential_chunk = current_chunk + (sep if current_chunk else "") + part
            
            if len(potential_chunk) <= max_chunk_size:
                current_chunk = potential_chunk
            else:
                if current_chunk:
                    chunks.append(current_chunk)
                
                # Si una sola palabra/parte es más grande que el máximo, bajamos de nivel en los separadores
                if len(part) > max_chunk_size:
                    chunks.extend(_split_recursive(part, current_seps[1:]))
                    current_chunk = ""
                else:
                    current_chunk = part
                    
        if current_chunk:
            chunks.append(current_chunk)
            
        return chunks

    # Fase 1: Dividir el texto respetando la estructura semántica
    base_chunks = _split_recursive(text, separators)
    
    # Fase 2: Aplicar el solapamiento (Overlap)
    final_chunks = []
    for i, chunk in enumerate(base_chunks):
        if i == 0:
            final_chunks.append(chunk)
        else:
            prev_chunk = base_chunks[i - 1]
            # Tomamos los últimos 'overlap_size' caracteres del chunk anterior
            overlap_text = prev_chunk[-overlap_size:] if len(prev_chunk) >= overlap_size else prev_chunk
            
            # Construimos el nuevo chunk con el contexto previo
            combined_chunk = overlap_text + " " + chunk
            final_chunks.append(combined_chunk)
            
    return final_chunks



    

In [18]:
mis_chunks = recursive_text_splitter(text_pdf, max_chunk_size=150, overlap_size=30)
    
print(f"Total de chunks generados: {len(mis_chunks)}\n")
for idx, c in enumerate(mis_chunks):
    print(f"--- CHUNK {idx + 1} ({len(c)} caracteres) ---")
    print(c)
    print("-" * 30)

    

Total de chunks generados: 88

--- CHUNK 1 (140 caracteres) ---
# A Tutorial on Principal Component Analysis

Jonathon Shlens*
Google Research
Mountain View, CA 94043

(Dated: April 7, 2014; Version 3.02)
------------------------------
--- CHUNK 2 (137 caracteres) ---
: April 7, 2014; Version 3.02) Principal component analysis (PCA) is a mainstay of modern data analysis - a black box that is widely used
------------------------------
--- CHUNK 3 (139 caracteres) ---
 black box that is widely used but (sometimes) poorly understood. The goal of this paper is to dispel the magic behind this black box. This
------------------------------
--- CHUNK 4 (136 caracteres) ---
ic behind this black box. This manuscript focuses on building a solid intuition for how and why principal component analysis works. This
------------------------------
--- CHUNK 5 (138 caracteres) ---
component analysis works. This manuscript crystallizes this knowledge by deriving from simple intuitions, the mathematics be

In [19]:
def obtener_embeddings(lista_de_textos: list[str]) -> list[list[float]]:
    """
    Recibe una lista de strings y devuelve una lista con sus respectivos vectores de embedding.
    """
    if not lista_de_textos:
        return []

    # Enviamos toda la lista a la API en un solo llamado
    respuesta = client.embeddings.create(
        input=lista_de_textos,
        model="text-embedding-3-small"  # "Nombre de implementación" de tu captura
    )
    
    # Extraemos el vector (embedding) de cada elemento en el orden correcto
    embeddings = [objeto.embedding for objeto in respuesta.data]
    
    return embeddings

In [20]:
embeddings = obtener_embeddings(mis_chunks)

In [22]:
print(embeddings[0])

[-0.0172882080078125, -0.015899658203125, 0.034759521484375, 0.0009717941284179688, -0.0018568038940429688, -0.00315093994140625, -0.006946563720703125, -0.0217437744140625, -0.030853271484375, 0.03851318359375, 0.041778564453125, -0.025177001953125, 0.01114654541015625, -0.00829315185546875, 0.019561767578125, -0.0067138671875, 0.037689208984375, 0.01110076904296875, -0.01971435546875, 0.03704833984375, 0.00167083740234375, -0.0258941650390625, 0.0271759033203125, 0.0223846435546875, -0.05596923828125, -0.025238037109375, 0.0305938720703125, 0.053741455078125, 0.03033447265625, -0.05426025390625, -0.037078857421875, -0.0245361328125, -0.048065185546875, 0.03826904296875, -0.04669189453125, 0.01222991943359375, -0.0201263427734375, -0.01082611083984375, -0.036346435546875, 0.0020618438720703125, -0.0184326171875, 0.00795745849609375, 0.00360870361328125, 0.025115966796875, -0.006793975830078125, 0.0064544677734375, -0.01898193359375, -0.006420135498046875, -0.037994384765625, 0.0475769

In [24]:
prompt = 'Hola, que es el PCA y cuales son sus ventajas'
prompt_embedding = obtener_embeddings(prompt)

In [25]:
print(prompt_embedding)

[[-0.01435089111328125, -0.005035400390625, 0.06158447265625, 0.048797607421875, 0.0242156982421875, 0.060455322265625, -0.041595458984375, 0.017608642578125, 0.0065155029296875, 0.01363372802734375, 0.04443359375, -0.0161285400390625, 0.0200042724609375, -0.036529541015625, -0.0003142356872558594, 0.0166015625, 0.0291595458984375, 0.0216522216796875, 0.00588226318359375, 0.0253448486328125, -0.037353515625, 0.01139068603515625, -0.024932861328125, 0.017181396484375, -0.01558685302734375, -0.0303802490234375, -0.01702880859375, 0.00021207332611083984, -0.01227569580078125, -0.0021915435791015625, 0.00737762451171875, -0.032257080078125, -0.026031494140625, -0.01456451416015625, -0.0144500732421875, 0.0242462158203125, -0.0310211181640625, 0.060394287109375, -0.03582763671875, 0.047637939453125, 0.01483154296875, 0.02288818359375, -0.0212249755859375, 0.0030117034912109375, 0.0191802978515625, -0.031524658203125, -0.0147857666015625, 0.0021419525146484375, -0.04010009765625, 0.048950195

### IVF

In [36]:
res = vf_search(embeddings,prompt_embedding,mis_chunks)

In [47]:
print(res)

['better understanding of PCA as well as the when, the how and the why of applying this technique.', 'g assumptions. My hope is that a thorough understanding of PCA provides a foundation for\napproaching the fields of machine learning and dimensional\nreduction.', ': April 7, 2014; Version 3.02) Principal component analysis (PCA) is a mainstay of modern data analysis - a black box that is widely used', '\n## I. INTRODUCTION Principal component analysis (PCA) is a standard tool in mod-\nern data analysis - in diverse fields from neuroscience to com-', 'ctures that often underlie it. The goal of this tutorial is to provide both an intuitive feel for\nPCA, and a thorough discussion of this topic. We will begin']


### Creacion del prompt (prompt engineer)

In [53]:
context = "\n\n".join(res)

tuning_prompt = f"""
Eres un asistente experto. Responde utilizando únicamente la información proporcionada.

Pregunta del usuario:
{prompt}

Contexto recuperado:
{context}

Instrucciones:
- Responde en el mismo idioma del contexto.
- Organiza la información para que sea clara y fácil de entender.
- No agregues información que no esté presente en el contexto.
- Si la respuesta no está en el contexto, indica que no hay suficiente información.
"""

### Funcion para preguntar al LLM

In [54]:
def preguntar_llm(prompt: str) -> str:
    try:
        respuesta = client.chat.completions.create(
            model=DEPLOYMENT, 
            messages=[
                {"role": "system", "content": "Eres un asistente de IA servicial y experto en datos."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7 
        )
        return respuesta.choices[0].message.content
    except Exception as e:
        return f"Ocurrió un error al conectar con Azure OpenAI: {e}"

In [55]:
response = preguntar_llm(tunnig_prompt)

In [56]:
print(response)

**Introducción a la Análisis de Componentes Principales (PCA): Una Guía Integral**

**Fecha de Versión:** 7 de abril de 2014; Versión 3.02

**I. Introducción**

El análisis de componentes principales (PCA) es una herramienta esencial en el análisis de datos modernos, con aplicaciones diversas que van desde la ciencia del cerebro hasta las complejas tareas de la industria. El objetivo de esta guía es proporcionar una comprensión intuitiva de PCA, así como una discusión detallada del tema. Comenzaremos con un sentimiento general de la utilidad de PCA y luego profundizaremos en los detalles técnicos, asegurándonos de cubrir los aspectos clave: cuándo usarlo, cómo aplicarlo, y por qué es una herramienta tan poderosa.

**Requisitos**

- Conocer las bases matemáticas necesarias para PCA.
- Entender la naturaleza de los datos y su distribución.
- Familiaridad con el propósito del análisis de datos en diversas disciplinas.

**II. El Proceso de PCA**

PCA es una técnica de reducción de dimensio